# Movie Recommender — Embedding + Attention
Content-based recommendation system: plot description → meaning vector → similar movies.

## 1. Imports

In [ ]:
import ast
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Attention, GlobalAveragePooling1D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load Data

In [ ]:
df = pd.read_csv('tmdb_5000_movies.csv')
df.head(1)

In [ ]:
new_df = df[['original_title', 'genres', 'overview']]
new_df.head(1)

## 3. Data Cleaning
Parse the `genres` column (stringified JSON) into a single primary genre label, and drop rows with missing text.

In [ ]:
def extract_primary_genre(genres_str):
    try:
        genres_list = ast.literal_eval(genres_str)
        if len(genres_list) > 0:
            return genres_list[0]['name']
    except (ValueError, SyntaxError):
        pass
    return None

df_clean = new_df.copy()
df_clean['genre'] = df_clean['genres'].apply(extract_primary_genre)

df_clean = df_clean.dropna(subset=['genre', 'overview'])
df_clean = df_clean[df_clean['overview'].str.strip() != '']
df_clean = df_clean.reset_index(drop=True)

print(f"Rows before cleaning: {len(new_df)}")
print(f"Rows after cleaning: {len(df_clean)}")
print(f"Number of unique genres: {df_clean['genre'].nunique()}")
df_clean[['original_title', 'genre', 'overview']].head(5)

## 4. Tokenization + Padding
Convert overview text into fixed-length integer sequences, and encode genre labels as integers.

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df_clean['overview'])

sequences = tokenizer.texts_to_sequences(df_clean['overview'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_clean['genre'])
num_classes = len(label_encoder.classes_)

print("Vocabulary size used:", min(VOCAB_SIZE, len(tokenizer.word_index) + 1))
print("Shape of X (movies x max words):", X.shape)
print("Shape of y (movies,):", y.shape)
print("Number of genre classes:", num_classes)

## 5. Build the Model

- **Embedding** — turns each word ID into a learned 32-number vector. `mask_zero=True` tells every
  downstream layer to ignore padding (critical — without this, padding dominates the pooled vector).
- **Attention** — self-attention over the word embeddings, learns which words matter most.
- **GlobalAveragePooling1D** — averages the (masked) attended word vectors into one "meaning vector"
  per movie. This is the vector we reuse for recommendations.
- **Dropout** — regularization, discourages the model from leaning on just a few dimensions.
- **Dense (softmax)** — predicts genre. We only use this as a training signal; the real payoff is
  the meaning vector produced right before it.

In [ ]:
EMBED_DIM = 32

i = Input(shape=(MAX_LEN,), name='overview_input')
E = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, name='word_embedding', mask_zero=True)(i)
A = Attention(name='attention')([E, E])
pooled = GlobalAveragePooling1D(name='meaning_vector')(A)
pooled_drop = Dropout(0.4)(pooled)
D = Dense(num_classes, activation='softmax')(pooled_drop)

model = Model(inputs=i, outputs=D)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Train
Early stopping restores the best weights (lowest validation loss) automatically, so we don't have to guess the right number of epochs.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop]
)

## 7. Extract Meaning Vectors

We don't care about the genre prediction itself — we cut the model open at the `meaning_vector`
layer and read out its output for every movie. That's the reusable 32-number "fingerprint" per movie.

In [ ]:
vector_model = Model(inputs=model.input, outputs=model.get_layer('meaning_vector').output)
movie_vectors = vector_model.predict(X, batch_size=64)

print("Shape of all movie vectors:", movie_vectors.shape)
print("Example vector (first 10 numbers of movie 0):", movie_vectors[0][:10])

## 8. Recommender Function
Cosine similarity between meaning vectors — no retraining needed, just a lookup.

In [ ]:
similarity_matrix = cosine_similarity(movie_vectors)

def recommend(title, top_n=5):
    matches = df_clean[df_clean['original_title'].str.lower() == title.lower()]
    if matches.empty:
        print(f"'{title}' not found in dataset.")
        return

    idx = matches.index[0]
    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:top_n]

    print(f"Movies similar to '{df_clean['original_title'].iloc[idx]}' ({df_clean['genre'].iloc[idx]}):\n")
    for i, score in scores:
        print(f"  {df_clean['original_title'].iloc[i]}  —  similarity: {score:.2f}  ({df_clean['genre'].iloc[i]})")

recommend("Avatar")

## 9. Save Model + Data for the Streamlit App

Saves everything `app.py` needs: the trained model, tokenizer, and the precomputed
vectors/similarity matrix/titles/genres (so the app loads instantly, no retraining).

In [ ]:
model.save('movie_model.keras')

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open('movie_data.pkl', 'wb') as f:
    pickle.dump({
        'movie_vectors': movie_vectors,
        'similarity_matrix': similarity_matrix,
        'titles': df_clean['original_title'].values,
        'genres': df_clean['genre'].values
    }, f)

print("Saved: movie_model.keras, tokenizer.pkl, movie_data.pkl")